# FIN 4000 — Chapter 16: Managing Bond Portfolios
*Duration as interest-rate sensitivity, and why immunization depends on it*

**ANSWER KEY — fully worked, instructor reference**

This notebook is the Julia companion to the Chapter 16 Excel template — same data, same formulas, same numbers. Where the Excel workshop has you build a formula in a cell, this notebook has you build it in a line of Julia instead.

## Step 1 — Entry Quiz *(3 minutes, individually)*

1. In plain terms, what does duration measure?
2. Why does a bond with a higher coupon rate have a lower duration than an otherwise-identical bond with a lower coupon?
3. What does it mean to “immunize” a bond portfolio against interest-rate risk?

---

In [ ]:
using Statistics, Printf

## Step 2 — Worked Example *(10 minutes, instructor-led)*

A 3-year, $1,000 face-value, 6% annual-coupon bond, priced at an 8% yield (price = $948.46, from Chapter 14's method).

In [ ]:
using Statistics
face, coupon_rate, n, ytm = 1000, 0.06, 3, 0.08
cash_flows = [60, 60, 1060]
periods = [1, 2, 3]

In [ ]:
pv = cash_flows ./ (1 + ytm) .^ periods
price = sum(pv)
mac_duration = sum(periods .* pv) / price
[(t=periods[i], CF=cash_flows[i], PV=round(pv[i]; digits=2), t_x_PV=round(periods[i] * pv[i]; digits=2)) for i in eachindex(periods)]

In [ ]:
println(string("Price=\$", @sprintf("%.2f", price), "  Macaulay Duration=", @sprintf("%.2f", mac_duration), " years"))

*Σ(t×PV) = 2,682.84; Price = $948.46. Macaulay Duration = 2,682.84 ÷ 948.46 = 2.83 years — shorter than the bond's 3-year maturity, since coupons return some cash before the final year.*

## Step 3 — Pair Work *(12 minutes, with a partner)*

**1.** Compute Modified Duration from the Macaulay Duration above (Modified Duration = Macaulay Duration ÷ (1 + YTM)).

In [ ]:
mod_duration = mac_duration / (1 + ytm)
println(string("Modified Duration = ", @sprintf("%.2f", mod_duration)))

**2.** Use Modified Duration to estimate the % price change and the new price if the yield rises from 8% to 9%.

In [ ]:
delta_y = 0.01
est_pct_change = -mod_duration * delta_y
est_price = price * (1 + est_pct_change)
println(string("Estimated % change=", @sprintf("%.2f%%", (est_pct_change) * 100), "  Estimated new price=\$", @sprintf("%.2f", est_price)))

**3.** Reprice the bond exactly at 9% (same PV approach as Chapter 14) and compare it to your duration estimate. Which is higher, and what does that gap tell you about convexity?

In [ ]:
new_ytm = ytm + delta_y
exact_price = sum(cash_flows ./ (1 + new_ytm) .^ periods)
convexity_gap = exact_price - est_price
println("Exact price=", @sprintf("%.2f", exact_price), "  Estimate=", @sprintf("%.2f", est_price), "  Gap (convexity)=", @sprintf("%.2f", convexity_gap))

## Step 4 — Julia Pass *(7 minutes)*

- The Cash Flow / PV / t×PV table, with SUM() formulas feeding a Macaulay Duration cell
- Modified Duration and the %-price-change estimate, built from that cell
- An exact reprice at the new yield using PV(), placed right next to the duration estimate for comparison

In [ ]:
function bond_duration(face, coupon_rate, n, ytm)
    periods = collect(1:n)
    cash_flows = fill(face * coupon_rate, n)
    cash_flows[end] += face
    pv = cash_flows ./ (1 + ytm) .^ periods
    price = sum(pv)
    macaulay = sum(periods .* pv) / price
    return Dict(:price => price, :macaulay => macaulay, :modified => macaulay / (1 + ytm))
end

bond_duration(1000, 0.06, 3, 0.08)

## Step 5 — Discussion *(3-5 minutes, no calculation)*

> Two bonds have the same maturity, but Bond A pays a higher coupon than Bond B. Which bond has the higher duration, and why does that matter for someone trying to immunize a portfolio against interest-rate risk?